## Different Retriever Techniques

In [1]:
from pathlib import Path
import getpass #getpass securely reads sensitive input, such as passwords or API keys, without displaying them on the screen.
import os #provides functions for interacting with the operating system, such as reading environment variables and managing files.
import shutil #shutil provides high-level file operations such as copying, moving, and deleting files and directories.

import numpy as np
import pandas as pd

from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

from langchain_chroma import Chroma

C:\Users\anilp\AppData\Local\Temp\ipykernel_13052\1183619124.py:9: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader
c:\Mine\AI\Full Stack Gen AI  BootCamp (KrishNaik)\Practicals\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
huggingfacehub_api_token = os.getenv("huggingfacehub_api_token")

In [3]:
from langchain_huggingface import HuggingFaceEndpointEmbeddings

embeddings = HuggingFaceEndpointEmbeddings(
    model="BAAI/bge-small-en-v1.5",
    task="feature-extraction",
)

In [4]:
DATA_DIR = Path(
    r"C:\Mine\AI\Full Stack Gen AI  BootCamp (KrishNaik)\Practicals"
    r"\Class-36-29-July-2026_Retriever\data"
)

preferred_pdf = DATA_DIR / "llama2-research-paper.pdf"

if preferred_pdf.exists():
    PDF_PATH = preferred_pdf
else:
    # Automatically find the PDF if its filename is slightly different
    available_pdfs = sorted(DATA_DIR.glob("*.pdf"))

    if len(available_pdfs) == 1:
        PDF_PATH = available_pdfs[0]
    elif len(available_pdfs) == 0:
        raise FileNotFoundError(
            f"No PDF file was found inside:\n{DATA_DIR}"
        )
    else:
        raise RuntimeError(
            "Multiple PDF files were found. Please set PDF_PATH manually.\n"
            + "\n".join(str(path) for path in available_pdfs)
        )

print("PDF found:")
print(PDF_PATH)

PDF found:
C:\Mine\AI\Full Stack Gen AI  BootCamp (KrishNaik)\Practicals\Class-36-29-July-2026_Retriever\data\llama2-research-paper.pdf


In [5]:
loader=PyPDFLoader(str(PDF_PATH))

pages=loader.load()
print(f"Total Pages: {len(pages)}")

Total Pages: 77


In [6]:
print("First-page metadata:")
print(pages[0].metadata)

print("\nFirst 1,000 characters:")
print(pages[0].page_content[:1000])

First-page metadata:
{'producer': 'pdfTeX-1.40.25', 'creator': 'LaTeX with hyperref', 'creationdate': '2023-07-20T00:30:36+00:00', 'author': '', 'keywords': '', 'moddate': '2023-07-20T00:30:36+00:00', 'ptex.fullbanner': 'This is pdfTeX, Version 3.141592653-2.6-1.40.25 (TeX Live 2023) kpathsea version 6.3.5', 'subject': '', 'title': '', 'trapped': '/False', 'source': 'C:\\Mine\\AI\\Full Stack Gen AI  BootCamp (KrishNaik)\\Practicals\\Class-36-29-July-2026_Retriever\\data\\llama2-research-paper.pdf', 'total_pages': 77, 'page': 0, 'page_label': '1'}

First 1,000 characters:
Llama 2: Open Foundation and Fine-Tuned Chat Models
Hugo Touvron∗ Louis Martin† Kevin Stone†
Peter Albert Amjad Almahairi Yasmine Babaei Nikolay Bashlykov Soumya Batra
Prajjwal Bhargava Shruti Bhosale Dan Bikel Lukas Blecher Cristian Canton Ferrer Moya Chen
Guillem Cucurull David Esiobu Jude Fernandes Jeremy Fu Wenyin Fu Brian Fuller
Cynthia Gao Vedanuj Goswami Naman Goyal Anthony Hartshorn Saghar Hosseini Rui Hou
Haka

## MetaData (Customization)

In [7]:
def identify_section(paper_page: int) -> str:
    """
    Identify the major section of the Llama 2 paper
    using its printed PDF page number.
    """

    if 1 <= paper_page <= 2:
        return "front_matter"

    if 3 <= paper_page <= 4:
        return "introduction"

    if 5 <= paper_page <= 7:
        return "pretraining"

    if 8 <= paper_page <= 19:
        return "fine_tuning"

    if 20 <= paper_page <= 31:
        return "safety"

    if 32 <= paper_page <= 35:
        return "discussion"

    if paper_page == 36:
        return "conclusion"

    if 37 <= paper_page <= 45:
        return "references"

    if 46 <= paper_page <= 77:
        return "appendix"

    return "unknown"

In [8]:
for page_document in pages:
    # PyPDFLoader page index is normally zero-based
    page_index = int(page_document.metadata.get("page", 0))
    paper_page = page_index + 1

    page_document.metadata.update(
        {
            "paper": "Llama 2",
            "organization": "Meta",
            "year": 2023,
            "document_type": "research_paper",
            "paper_page": paper_page,
            "section": identify_section(paper_page),
            "access_level": "public",
        }
    )

In [9]:
for page_document in pages[:5]:
    print(page_document.metadata)

{'producer': 'pdfTeX-1.40.25', 'creator': 'LaTeX with hyperref', 'creationdate': '2023-07-20T00:30:36+00:00', 'author': '', 'keywords': '', 'moddate': '2023-07-20T00:30:36+00:00', 'ptex.fullbanner': 'This is pdfTeX, Version 3.141592653-2.6-1.40.25 (TeX Live 2023) kpathsea version 6.3.5', 'subject': '', 'title': '', 'trapped': '/False', 'source': 'C:\\Mine\\AI\\Full Stack Gen AI  BootCamp (KrishNaik)\\Practicals\\Class-36-29-July-2026_Retriever\\data\\llama2-research-paper.pdf', 'total_pages': 77, 'page': 0, 'page_label': '1', 'paper': 'Llama 2', 'organization': 'Meta', 'year': 2023, 'document_type': 'research_paper', 'paper_page': 1, 'section': 'front_matter', 'access_level': 'public'}
{'producer': 'pdfTeX-1.40.25', 'creator': 'LaTeX with hyperref', 'creationdate': '2023-07-20T00:30:36+00:00', 'author': '', 'keywords': '', 'moddate': '2023-07-20T00:30:36+00:00', 'ptex.fullbanner': 'This is pdfTeX, Version 3.141592653-2.6-1.40.25 (TeX Live 2023) kpathsea version 6.3.5', 'subject': '', '

## Chunking (RecursiveCharacterText Splitter)

In [10]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200,
    add_start_index=True,
)

chunks = text_splitter.split_documents(pages)

print(f"Total pages: {len(pages)}")
print(f"Total chunks: {len(chunks)}")

Total pages: 77
Total chunks: 343


In [11]:
## Capture the chunk number and paper page in the chunk metadata
#chunk_number is just the loop index that enumerate() produces enumerate(chunks) yields (0, chunks[0]), (1, chunks[1]), (2, chunks[2]), and so on
for chunk_number, chunk in enumerate(chunks):
    paper_page = chunk.metadata.get("paper_page", "unknown")

    chunk.metadata["chunk_id"] = (
        f"llama2-page-{paper_page}-chunk-{chunk_number}"
    )

In [12]:
print("Chunk content:")
print(chunks[0].page_content[:1100])

print("\nChunk metadata:")
print(chunks[0].metadata)

Chunk content:
Llama 2: Open Foundation and Fine-Tuned Chat Models
Hugo Touvron∗ Louis Martin† Kevin Stone†
Peter Albert Amjad Almahairi Yasmine Babaei Nikolay Bashlykov Soumya Batra
Prajjwal Bhargava Shruti Bhosale Dan Bikel Lukas Blecher Cristian Canton Ferrer Moya Chen
Guillem Cucurull David Esiobu Jude Fernandes Jeremy Fu Wenyin Fu Brian Fuller
Cynthia Gao Vedanuj Goswami Naman Goyal Anthony Hartshorn Saghar Hosseini Rui Hou
Hakan Inan Marcin Kardas Viktor Kerkez Madian Khabsa Isabel Kloumann Artem Korenev
Punit Singh Koura Marie-Anne Lachaux Thibaut Lavril Jenya Lee Diana Liskovich
Yinghai Lu Yuning Mao Xavier Martinet Todor Mihaylov Pushkar Mishra
Igor Molybog Yixin Nie Andrew Poulton Jeremy Reizenstein Rashi Rungta Kalyan Saladi
Alan Schelten Ruan Silva Eric Michael Smith Ranjan Subramanian Xiaoqing Ellen Tan Binh Tang
Ross Taylor Adina Williams Jian Xiang Kuan Puxin Xu Zheng Yan Iliyan Zarov Yuchen Zhang
Angela Fan Melanie Kambadur Sharan Narang Aurelien Rodriguez Robert Stojni

## Embeddings

In [13]:
test_vector=embeddings.embed_query("What is Llama 2?")

print(f"Embedding dimenesions:{len(test_vector)}")
print(f"Embedding vector (first 10 values): {test_vector[:10]}")

Embedding dimenesions:384
Embedding vector (first 10 values): [-0.08428332209587097, -0.024261441081762314, -0.015151201747357845, 0.03341759741306305, 0.04084879532456398, -0.0009922870667651296, -0.010198874399065971, 0.008071157149970531, -0.01886812411248684, -0.02917918935418129]


In [14]:
persistDirectoryPath=Path(
r"C:\Mine\AI\Full Stack Gen AI  BootCamp (KrishNaik)\Practicals"
r"\Class-36-29-July-2026_Retriever\PersistDirectory")

PERSIST_DIRECTORY = persistDirectoryPath / "chroma_llama2_retriever"

# Set this to False when you want to reuse the existing index.
REBUILD_INDEX = True

if REBUILD_INDEX and PERSIST_DIRECTORY.exists():
    shutil.rmtree(
        PERSIST_DIRECTORY,
        ignore_errors=True
    )

In [15]:
vectorstore=Chroma.from_documents(
    documents=chunks,
    embedding=embeddings,
    collection_name="llama2_retriever",
    persist_directory=str(PERSIST_DIRECTORY),
    collection_configuration={"hnsw":{"space": "cosine"}})

print("Vector Store created Successfully")
print(f"Stored Chunks: {len(chunks)}")
print(f"Vector Store Path: {PERSIST_DIRECTORY}")

Vector Store created Successfully
Stored Chunks: 343
Vector Store Path: C:\Mine\AI\Full Stack Gen AI  BootCamp (KrishNaik)\Practicals\Class-36-29-July-2026_Retriever\PersistDirectory\chroma_llama2_retriever


In [16]:
# Load the existing Chroma collection
vector_store = Chroma(
    collection_name="llama2_retriever",
    embedding_function=embeddings,
    persist_directory=str(PERSIST_DIRECTORY),
)

print("Existing vector store loaded successfully.")
print(f"Persist directory: {PERSIST_DIRECTORY}")

Existing vector store loaded successfully.
Persist directory: C:\Mine\AI\Full Stack Gen AI  BootCamp (KrishNaik)\Practicals\Class-36-29-July-2026_Retriever\PersistDirectory\chroma_llama2_retriever


In [17]:
def display_documents(
    documents,
    max_characters: int = 700
) -> None:
    """
    Display retrieved LangChain Document objects clearly.
    """

    if not documents:
        print("No documents were returned.")
        return

    for rank, document in enumerate(documents, start=1):
        metadata = document.metadata

        print("=" * 90)
        print(f"RANK: {rank}")
        print(f"PAPER PAGE: {metadata.get('paper_page')}")
        print(f"SECTION: {metadata.get('section')}")
        print(f"CHUNK ID: {metadata.get('chunk_id')}")
        print(f"SOURCE: {metadata.get('source')}")
        print("-" * 90)
        print(document.page_content[:max_characters])
        print()

In [18]:
similarity_retriever = vector_store.as_retriever(
    search_type="similarity",
    search_kwargs={
        "k": 4
    }
)

In [19]:
query = "What model sizes of Llama 2 were released?"
similarity_documents = similarity_retriever.invoke(query)
display_documents(similarity_documents)

RANK: 1
PAPER PAGE: 77
SECTION: appendix
CHUNK ID: llama2-page-77-chunk-338
SOURCE: C:\Mine\AI\Full Stack Gen AI  BootCamp (KrishNaik)\Practicals\Class-36-29-July-2026_Retriever\data\llama2-research-paper.pdf
------------------------------------------------------------------------------------------
A.7 Model Card
Table 52 presents a model card (Mitchell et al., 2018; Anil et al., 2023) that summarizes details of the models.
Model Details
Model DevelopersMeta AI
Variations Llama 2comes in a range of parameter sizes—7B, 13B, and 70B—as well as
pretrained and fine-tuned variations.
Input Models input text only.
Output Models generate text only.
Model ArchitectureLlama 2isanauto-regressivelanguagemodelthatusesanoptimizedtransformer
architecture. The tuned versions use supervised fine-tuning (SFT) and reinforce-
ment learning with human feedback (RLHF) to align to human preferences for
helpfulness and safety.
Model Dates Llama 2was trained between January 2023 and July 2023.
Status This is 

# MMR Search Type

***MMR (Maximal Marginal Relevance) retrieval — used in vector stores (LangChain, etc.) when you want results that are both relevant AND diverse, not just the top-k most similar chunks.***

**Flow**
1. Fetch 20 similar chunks (fetch_k)
2. From those 20, pick the final 4 (k) by balancing similarity to the query with dissimilarity to already-picked chunks, weighted by lambda_mult

***Note***: For RAG, 0.5 is a common default because you want the LLM to see complementary context, not the same sentence three times.



In [20]:
mmr_retriever = vector_store.as_retriever(
    search_type="mmr",
    search_kwargs={
        "k": 4,
        "fetch_k": 20,
        "lambda_mult": 0.5,
    }
)

In [21]:
query = "How was Llama 2-Chat trained and aligned?"

mmr_documents = mmr_retriever.invoke(query)

display_documents(mmr_documents)

RANK: 1
PAPER PAGE: 8
SECTION: fine_tuning
CHUNK ID: llama2-page-8-chunk-29
SOURCE: C:\Mine\AI\Full Stack Gen AI  BootCamp (KrishNaik)\Practicals\Class-36-29-July-2026_Retriever\data\llama2-research-paper.pdf
------------------------------------------------------------------------------------------
are from OpenAI (2023). Results for the PaLM model are from Chowdhery et al. (2022). Results for the
PaLM-2-L are from Anil et al. (2023).
3 Fine-tuning
Llama 2-Chat is the result of several months of research and iterative applications of alignment techniques,
including both instruction tuning and RLHF, requiring significant computational and annotation resources.
In this section, we report on our experiments and findings using supervised fine-tuning (Section 3.1), as
well as initial and iterative reward modeling (Section 3.2.2) and RLHF (Section 3.2.3). We also share a
new technique, Ghost Attention (GAtt), which we find helps control dialogue flow over multiple turns
(Section 3.3). See Se

## Compare between Cosine Similarity And MMR

In [22]:
query = "How was Llama 2-Chat trained and aligned?"

similarity_retriever = vector_store.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 4}
)

mmr_retriever = vector_store.as_retriever(
    search_type="mmr",
    search_kwargs={
        "k": 4,
        "fetch_k": 20,
        "lambda_mult": 0.5,
    }
)

similarity_results = similarity_retriever.invoke(query)
mmr_results = mmr_retriever.invoke(query)

In [23]:
print("Similarity Search results:")
for document in similarity_results:
    print(
        document.metadata.get("paper_page"),
        document.metadata.get("section"),
        document.metadata.get("chunk_id"),
    )

print("\nMMR results:")
for document in mmr_results:
    print(
        document.metadata.get("paper_page"),
        document.metadata.get("section"),
        document.metadata.get("chunk_id"),
    )

Similarity Search results:
8 fine_tuning llama2-page-8-chunk-29
3 introduction llama2-page-3-chunk-9
5 pretraining llama2-page-5-chunk-14
1 front_matter llama2-page-1-chunk-1

MMR results:
8 fine_tuning llama2-page-8-chunk-29
3 introduction llama2-page-3-chunk-9
11 fine_tuning llama2-page-11-chunk-45
56 appendix llama2-page-56-chunk-247


**Above Comparision Explanation**

***What actually happened***

First two picks are identical in both — the most relevant chunks (page 8 fine_tuning, page 3 introduction) show up in both lists. MMR always picks the most relevant chunk first, so overlap at the top is expected.

Picks 3 and 4 diverge — this is MMR earning its keep:

- Similarity search grabbed page 5 (pretraining) and page 1 (front_matter) — likely because those chunks had high cosine similarity to the query, but they may cover overlapping ground with the top picks or with each other.

- MMR looked at what was already picked (chunks from pages 8 and 3) and asked: "what else is relevant BUT covers different ground?" It found:
    - Page 11 chunk 45 — another fine_tuning chunk, but from a different page (so likely different subtopic within fine-tuning — maybe a different technique or dataset)
    - Page 56 chunk 247 — jumps all the way to the appendix, probably pulling in supplementary details/tables the main sections didn't cover

## Now we have used score_threshold
- score_threshold: Picks chunks those are matched with provided threshold.

In [24]:
threshold_retriever = vector_store.as_retriever(
    search_type="similarity_score_threshold",
    search_kwargs={
        "k": 10,
        "score_threshold": 0.64,
    }
)

In [25]:
query = "What safety techniques were used for Llama 2-Chat?"

threshold_documents = threshold_retriever.invoke(query)

display_documents(threshold_documents)

RANK: 1
PAPER PAGE: 4
SECTION: introduction
CHUNK ID: llama2-page-4-chunk-13
SOURCE: C:\Mine\AI\Full Stack Gen AI  BootCamp (KrishNaik)\Practicals\Class-36-29-July-2026_Retriever\data\llama2-research-paper.pdf
------------------------------------------------------------------------------------------
Solaiman et al., 2023). Testing conducted to date has been in English and has not — and could not — cover
all scenarios. Therefore, before deploying any applications ofLlama 2-Chat, developers should perform
safety testing and tuning tailored to their specific applications of the model. We provide a responsible use
guide¶ and code examples‖ to facilitate the safe deployment ofLlama 2 and Llama 2-Chat. More details of
our responsible release strategy can be found in Section 5.3.
The remainder of this paper describes our pretraining methodology (Section 2), fine-tuning methodology
(Section 3), approach to model safety (Section 4), key observations and insights (Section 5), relevant related
wo

## Retrieve Chunks with their relevance score

In [26]:
query = "What safety techniques were used for Llama 2-Chat?"

scored_results = vector_store.similarity_search_with_relevance_scores(
    query=query,
    k=5,
)

for rank, (document, relevance_score) in enumerate(
    scored_results,
    start=1
):
    print("=" * 90)
    print(f"Rank: {rank}")
    print(f"Relevance score: {relevance_score:.4f}")
    print(f"Paper page: {document.metadata.get('paper_page')}")
    print(f"Section: {document.metadata.get('section')}")
    print(document.page_content[:500])

Rank: 1
Relevance score: 0.8044
Paper page: 4
Section: introduction
Solaiman et al., 2023). Testing conducted to date has been in English and has not — and could not — cover
all scenarios. Therefore, before deploying any applications ofLlama 2-Chat, developers should perform
safety testing and tuning tailored to their specific applications of the model. We provide a responsible use
guide¶ and code examples‖ to facilitate the safe deployment ofLlama 2 and Llama 2-Chat. More details of
our responsible release strategy can be found in Section 5.3.
The remainder of 
Rank: 2
Relevance score: 0.8044
Paper page: 10
Section: fine_tuning
tworesponsestoagivenpromptaresampledfromtwodifferentmodelvariants,andvaryingthetemperature
hyper-parameter. Inadditiontogivingparticipantsaforcedchoice,wealsoaskannotatorstolabelthedegree
to which they prefer their chosen response over the alternative: either their choice issignificantly better, better,
slightly better, ornegligibly better/ unsure.
For our coll

In [27]:
metric_query = "How was reinforcement learning with human feedback used?"

candidate_documents = vector_store.similarity_search(
    metric_query,
    k=6,
)

candidate_texts = [
    document.page_content
    for document in candidate_documents
]

print(f"Candidate chunks selected: {len(candidate_texts)}")

Candidate chunks selected: 6


In [28]:
query_vector = np.asarray(
    embeddings.embed_query(metric_query),
    dtype=np.float64,
)

document_vectors = np.asarray(
    embeddings.embed_documents(candidate_texts),
    dtype=np.float64,
)

print("Query-vector shape:", query_vector.shape)
print("Document-vectors shape:", document_vectors.shape)

Query-vector shape: (384,)
Document-vectors shape: (6, 384)


In [29]:
def cosine_similarity(
    vector_a: np.ndarray,
    vector_b: np.ndarray
) -> float:
    denominator = (
        np.linalg.norm(vector_a)
        * np.linalg.norm(vector_b)
    )

    if denominator == 0:
        return 0.0

    return float(
        np.dot(vector_a, vector_b) / denominator
    )


def euclidean_distance(
    vector_a: np.ndarray,
    vector_b: np.ndarray
) -> float:
    return float(
        np.linalg.norm(vector_a - vector_b)
    )


def dot_product(
    vector_a: np.ndarray,
    vector_b: np.ndarray
) -> float:
    return float(
        np.dot(vector_a, vector_b)
    )

In [30]:
metric_rows = []

for document, document_vector in zip(
    candidate_documents,
    document_vectors
):
    metric_rows.append(
        {
            "paper_page": document.metadata.get("paper_page"),
            "section": document.metadata.get("section"),
            "chunk_id": document.metadata.get("chunk_id"),
            "cosine_similarity": cosine_similarity(
                query_vector,
                document_vector
            ),
            "euclidean_distance": euclidean_distance(
                query_vector,
                document_vector
            ),
            "dot_product": dot_product(
                query_vector,
                document_vector
            ),
            "preview": document.page_content[:100].replace(
                "\n",
                " "
            ),
        }
    )

metric_table = pd.DataFrame(metric_rows)

metric_table

,paper_page,section,chunk_id,cosine_similarity,euclidean_distance,dot_product,preview
0,9,fine_tuning,llama2-page-9-chunk-34,0.804664,0.625038,0.804664,"learning rate of2 × 10−5, a weight decay of 0...."
1,10,fine_tuning,llama2-page-10-chunk-35,0.776110,0.669163,0.776110,"sampled human preferences, whereby human annot..."
2,32,discussion,llama2-page-32-chunk-138,0.769533,0.678921,0.769533,"bility, seemed a somewhat shadowy field for th..."
3,2,front_matter,llama2-page-2-chunk-3,0.766612,0.683210,0.766612,Contents 1 Introduction 3 2 Pretraining 5 2.1 ...
4,15,fine_tuning,llama2-page-15-chunk-63,0.752870,0.703036,0.752870,We iteratively improve the policy by sampling ...
5,11,fine_tuning,llama2-page-11-chunk-42,0.747711,0.710337,0.747711,Each example consists of a prompt (including p...


In [31]:
metric_table.sort_values(
    by="cosine_similarity",
    ascending=False
)[
    [
        "paper_page",
        "section",
        "cosine_similarity",
        "preview",
    ]
]

,paper_page,section,cosine_similarity,preview
0,9,fine_tuning,0.804664,"learning rate of2 × 10−5, a weight decay of 0...."
1,10,fine_tuning,0.776110,"sampled human preferences, whereby human annot..."
2,32,discussion,0.769533,"bility, seemed a somewhat shadowy field for th..."
3,2,front_matter,0.766612,Contents 1 Introduction 3 2 Pretraining 5 2.1 ...
4,15,fine_tuning,0.752870,We iteratively improve the policy by sampling ...
5,11,fine_tuning,0.747711,Each example consists of a prompt (including p...


In [32]:
metric_table.sort_values(
    by="euclidean_distance",
    ascending=True
)[
    [
        "paper_page",
        "section",
        "euclidean_distance",
        "preview",
    ]
]

,paper_page,section,euclidean_distance,preview
0,9,fine_tuning,0.625038,"learning rate of2 × 10−5, a weight decay of 0...."
1,10,fine_tuning,0.669163,"sampled human preferences, whereby human annot..."
2,32,discussion,0.678921,"bility, seemed a somewhat shadowy field for th..."
3,2,front_matter,0.683210,Contents 1 Introduction 3 2 Pretraining 5 2.1 ...
4,15,fine_tuning,0.703036,We iteratively improve the policy by sampling ...
5,11,fine_tuning,0.710337,Each example consists of a prompt (including p...


In [33]:
metric_table.sort_values(
    by="dot_product",
    ascending=False
)[
    [
        "paper_page",
        "section",
        "dot_product",
        "preview",
    ]
]

,paper_page,section,dot_product,preview
0,9,fine_tuning,0.804664,"learning rate of2 × 10−5, a weight decay of 0...."
1,10,fine_tuning,0.776110,"sampled human preferences, whereby human annot..."
2,32,discussion,0.769533,"bility, seemed a somewhat shadowy field for th..."
3,2,front_matter,0.766612,Contents 1 Introduction 3 2 Pretraining 5 2.1 ...
4,15,fine_tuning,0.752870,We iteratively improve the policy by sampling ...
5,11,fine_tuning,0.747711,Each example consists of a prompt (including p...


In [34]:
normalized_query_vector = (
    query_vector / np.linalg.norm(query_vector)
)

normalized_document_vectors = (
    document_vectors
    / np.linalg.norm(
        document_vectors,
        axis=1,
        keepdims=True
    )
)

normalized_dot_scores = (
    normalized_document_vectors
    @ normalized_query_vector
)

cosine_scores = np.asarray(
    [
        cosine_similarity(
            query_vector,
            document_vector
        )
        for document_vector in document_vectors
    ]
)

print("Cosine scores:")
print(cosine_scores)

print("\nDot product after normalization:")
print(normalized_dot_scores)

print(
    "\nAre they approximately equal?",
    np.allclose(
        cosine_scores,
        normalized_dot_scores,
        atol=1e-8,
    ),
)

Cosine scores:
[0.80466355 0.77611034 0.76953343 0.76661224 0.75287015 0.74771052]

Dot product after normalization:
[0.80466355 0.77611034 0.76953343 0.76661224 0.75287015 0.74771052]

Are they approximately equal? True


## Prefiltering 

In [35]:
finetuning_retriever = vector_store.as_retriever(
    search_type="similarity",
    search_kwargs={
        "k": 4,
        "filter": {
            "section":"fine_tuning"
        },
    }
)


In [42]:
query = "How was Llama 2-Chat aligned with human preferences?"
fine_tuning_documents = finetuning_retriever.invoke(query)
display_documents(fine_tuning_documents)

RANK: 1
PAPER PAGE: 11
SECTION: fine_tuning
CHUNK ID: llama2-page-11-chunk-45
SOURCE: C:\Mine\AI\Full Stack Gen AI  BootCamp (KrishNaik)\Practicals\Class-36-29-July-2026_Retriever\data\llama2-research-paper.pdf
------------------------------------------------------------------------------------------
this study, the role of reward signals is to learn human preference forLlama 2-Chat outputs rather than
any modeloutputs. However, in our experiments, we do not observe negative transfer from the open-source
preference datasets. Thus, we have decided to keep them in our data mixture, as they could enable better
generalization for the reward model and prevent reward hacking, i.e.Llama 2-Chat taking advantage of
some weaknesses of our reward, and so artificially inflating the score despite performing less well.
With training data available from different sources, we experimented with different mixing recipes for both
Helpfulness and Safety reward models to ascertain the best settings. After 

In [43]:
for document in fine_tuning_documents:
    assert document.metadata.get("section") == "fine_tuning"

    print("All returned documents are from the fine_tuning section.")

All returned documents are from the fine_tuning section.
All returned documents are from the fine_tuning section.
All returned documents are from the fine_tuning section.
All returned documents are from the fine_tuning section.


## Prefiltering with Multiple Conditions

In [44]:
filtered_retriever = vector_store.as_retriever(
    search_type="similarity",
    search_kwargs={
        "k": 4,
        "filter": {
            "$and":[
                {
                    "section": {
                        "$eq": "fine_tuning"
                    }
                },
                 {
                    "year": {
                        "$eq": 2023
                    }
                },
                {
                    "organization": {
                        "$eq": "Meta"
                    }
                },
            ]
        },
    }
)

In [45]:
query = "How was human preference data collected?"
filtered_documents = filtered_retriever.invoke(query)
display_documents(filtered_documents)

RANK: 1
PAPER PAGE: 10
SECTION: fine_tuning
CHUNK ID: llama2-page-10-chunk-35
SOURCE: C:\Mine\AI\Full Stack Gen AI  BootCamp (KrishNaik)\Practicals\Class-36-29-July-2026_Retriever\data\llama2-research-paper.pdf
------------------------------------------------------------------------------------------
sampled human preferences, whereby human annotators select which of two model outputs they prefer.
This human feedback is subsequently used to train a reward model, which learns patterns in the preferences
of the human annotators and can then automate preference decisions.
3.2.1 Human Preference Data Collection
Next, we collect human preference data for reward modeling. We chose a binary comparison protocol over
other schemes, mainly because it enables us to maximize the diversity of collected prompts. Still, other
strategies are worth considering, which we leave for future work.
Our annotation procedure proceeds as follows. We ask annotators to first write a prompt, then choose
between tw

## Prefilter

In [46]:
pre_filter = {
    "$and": [
        {
            "section": {
                "$eq": "fine_tuning"
            }
        },
        {
            "year": {
                "$eq": 2023
            }
        },
    ]
}

pre_filtered_retriever = vector_store.as_retriever(
    search_type="similarity",
    search_kwargs={
        "k": 4,
        "filter": pre_filter,
    }
)

pre_filtered_documents = pre_filtered_retriever.invoke(
    "How was the reward model trained?"
)

display_documents(pre_filtered_documents)

RANK: 1
PAPER PAGE: 10
SECTION: fine_tuning
CHUNK ID: llama2-page-10-chunk-40
SOURCE: C:\Mine\AI\Full Stack Gen AI  BootCamp (KrishNaik)\Practicals\Class-36-29-July-2026_Retriever\data\llama2-research-paper.pdf
------------------------------------------------------------------------------------------
3.2.2 Reward Modeling
The reward model takes a model response and its corresponding prompt (including contexts from previous
turns) as inputs and outputs a scalar score to indicate the quality (e.g., helpfulness and safety) of the model
generation. Leveraging such response scores as rewards, we can optimizeLlama 2-Chat during RLHF for
better human preference alignment and improved helpfulness and safety.
Others have found that helpfulness and safety sometimes trade off (Bai et al., 2022a), which can make it
challenging for a single reward model to perform well on both. To address this, we train two separate reward
models, one optimized for helpfulness (referred to asHelpfulness RM) and ano

## PostFiltering

In [49]:
unfiltered_retriever_candidates = vector_store.similarity_search(
    query="How was the reward model trained?", k=15)

In [51]:
post_filtered_documents = [
document
for document in unfiltered_retriever_candidates
if document.metadata.get("section") == "fine_tuning"
and document.metadata.get("year") == 2023
]

display_documents(post_filtered_documents)

RANK: 1
PAPER PAGE: 10
SECTION: fine_tuning
CHUNK ID: llama2-page-10-chunk-40
SOURCE: C:\Mine\AI\Full Stack Gen AI  BootCamp (KrishNaik)\Practicals\Class-36-29-July-2026_Retriever\data\llama2-research-paper.pdf
------------------------------------------------------------------------------------------
3.2.2 Reward Modeling
The reward model takes a model response and its corresponding prompt (including contexts from previous
turns) as inputs and outputs a scalar score to indicate the quality (e.g., helpfulness and safety) of the model
generation. Leveraging such response scores as rewards, we can optimizeLlama 2-Chat during RLHF for
better human preference alignment and improved helpfulness and safety.
Others have found that helpfulness and safety sometimes trade off (Bai et al., 2022a), which can make it
challenging for a single reward model to perform well on both. To address this, we train two separate reward
models, one optimized for helpfulness (referred to asHelpfulness RM) and ano

In [54]:
print("Prefilter Return Documents.",len(unfiltered_retriever_candidates))
print("Post-filtered Documents.",len(post_filtered_documents))

Prefilter Return Documents. 15
Post-filtered Documents. 11


In [55]:
import os
from typing import List

from pydantic import BaseModel, Field

from langchain_community.retrievers import BM25Retriever
from langchain_classic.retrievers.ensemble import EnsembleRetriever

from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser


In [56]:
def display_documents(
    documents,
    title: str = "Retrieved Documents",
    max_documents: int = 10,
    max_characters: int = 600,
) -> None:
    """Display retrieved LangChain Document objects."""

    print("\n" + "=" * 100)
    print(title)
    print("=" * 100)

    if not documents:
        print("No documents were returned.")
        return

    for rank, document in enumerate(
        documents[:max_documents],
        start=1,
    ):
        metadata = document.metadata

        print(f"\nRANK: {rank}")
        print(f"Paper page: {metadata.get('paper_page')}")
        print(f"Section: {metadata.get('section')}")
        print(f"Chunk ID: {metadata.get('chunk_id')}")
        print("-" * 100)
        print(document.page_content[:max_characters])

In [57]:
def deduplicate_documents(documents):
    """Remove duplicate retrieved chunks while preserving their order."""

    unique_documents = []
    seen_keys = set()

    for document in documents:
        key = (
            document.metadata.get("chunk_id")
            or (
                document.metadata.get("source"),
                document.metadata.get("page"),
                document.page_content,
            )
        )

        if key not in seen_keys:
            seen_keys.add(key)
            unique_documents.append(document)

    return unique_documents

## Sparse (BM25) Retrieval
- BM25 is advance version of TF\IDF,Ranking algorithm used in information Retrieval.
- BM25 doenst required Vector Store
- It works on Keywords Search
- Very Long Documents are normalised as well.(It adjust Lengthy documents)

***BM25 Flow***
```text
Reads Query
   ||
   Finds matching Documents
   ||
   Computes Term Importance
   ||
   Adjusts docs Length
   ||
   Ranks documents


***Note: It is used in VectorLess RAG***

In [58]:
from langchain_community.retrievers import BM25Retriever

In [59]:
bm25_retriever = BM25Retriever.from_documents(chunks)

In [60]:
bm25_retriever

BM25Retriever(vectorizer=<rank_bm25.BM25Okapi object at 0x000002C211CBC510>)

In [67]:
# Final number of results
bm25_retriever.k = 4
sparse_query = "Grouped-Query Attention GQA 70B"

In [68]:
sparse_documents = bm25_retriever.invoke(sparse_query)

In [69]:
display_documents(
    sparse_documents,
    title="Sparse Retrieval: BM25 Results",
)


Sparse Retrieval: BM25 Results

RANK: 1
Paper page: 6
Section: pretraining
Chunk ID: llama2-page-6-chunk-18
----------------------------------------------------------------------------------------------------
Training Data Params Context
Length
GQA Tokens LR
Llama 1 See Touvron et al.
(2023)
7B 2k ✗ 1.0T 3.0 × 10−4
13B 2k ✗ 1.0T 3.0 × 10−4
33B 2k ✗ 1.4T 1.5 × 10−4
65B 2k ✗ 1.4T 1.5 × 10−4
Llama 2 A new mix of publicly
available online data
7B 4k ✗ 2.0T 3.0 × 10−4
13B 4k ✗ 2.0T 3.0 × 10−4
34B 4k ✓ 2.0T 1.5 × 10−4
70B 4k ✓ 2.0T 1.5 × 10−4
Table 1:Llama 2 family of models.Token counts refer to pretraining data only. All models are trained with
a global batch-size of 4M tokens. Bigger models — 34B and 70B — use Grouped-Query Attention (GQA) for
improved inference scalability.
0 250 500 750 1000 1250 15

RANK: 2
Paper page: 48
Section: appendix
Chunk ID: llama2-page-48-chunk-220
----------------------------------------------------------------------------------------------------
BoolQ PIQA 

## Dense Retrieval
- It does Semantic Search

In [70]:
dense_retriever = vector_store.as_retriever(
    search_type="similarity",
    search_kwargs={
        "k": 4
    },
)

In [71]:
dense_query = (
    "How did Meta improve inference scalability "
    "for the largest Llama 2 models?"
)

dense_documents = dense_retriever.invoke(dense_query)

display_documents(
    dense_documents,
    title="Dense Retrieval: Vector Search Results",
)


Dense Retrieval: Vector Search Results

RANK: 1
Paper page: 3
Section: introduction
Chunk ID: llama2-page-3-chunk-9
----------------------------------------------------------------------------------------------------
the community to advance AI alignment research.
In this work, we develop and release Llama 2, a family of pretrained and fine-tuned LLMs,Llama 2 and
Llama 2-Chat, at scales up to 70B parameters. On the series of helpfulness and safety benchmarks we tested,
Llama 2-Chat models generally perform better than existing open-source models. They also appear to
be on par with some of the closed-source models, at least on the human evaluations we performed (see
Figures 1 and 3). We have taken measures to increase the safety of these models, using safety-specific data
annotation and tuning, as well as c

RANK: 2
Paper page: 13
Section: fine_tuning
Chunk ID: llama2-page-13-chunk-53
----------------------------------------------------------------------------------------------------
m

In [72]:
comparison_query = (
    "How did grouped-query attention improve "
    "Llama 2 inference scalability?"
)

sparse_results = bm25_retriever.invoke(comparison_query)
dense_results = dense_retriever.invoke(comparison_query)

display_documents(
    sparse_results,
    title="BM25 Results",
    max_documents=4,
)

display_documents(
    dense_results,
    title="Dense Vector Results",
    max_documents=4,
)


BM25 Results

RANK: 1
Paper page: 4
Section: introduction
Chunk ID: llama2-page-4-chunk-12
----------------------------------------------------------------------------------------------------
1. Llama 2, an updated version ofLlama 1, trained on a new mix of publicly available data. We also
increased the size of the pretraining corpus by 40%, doubled the context length of the model, and
adopted grouped-query attention (Ainslie et al., 2023). We are releasing variants ofLlama 2 with
7B, 13B, and 70B parameters. We have also trained 34B variants, which we report on in this paper
but are not releasing.§
2. Llama 2-Chat, a fine-tuned version ofLlama 2 that is optimized for dialogue use cases. We release
variants of this model with 7B, 13B, and 70B parameters as well.
We believe that the

RANK: 2
Paper page: 47
Section: appendix
Chunk ID: llama2-page-47-chunk-217
----------------------------------------------------------------------------------------------------
attention (MHA) models grow 

In [73]:
print("SPARSE RESULTS")
for rank, document in enumerate(sparse_results, start=1):
    print(
        rank,
        document.metadata.get("paper_page"),
        document.metadata.get("chunk_id"),
    )

print("\nDENSE RESULTS")
for rank, document in enumerate(dense_results, start=1):
    print(
        rank,
        document.metadata.get("paper_page"),
        document.metadata.get("chunk_id"),
    )

SPARSE RESULTS
1 4 llama2-page-4-chunk-12
2 47 llama2-page-47-chunk-217
3 54 llama2-page-54-chunk-240
4 6 llama2-page-6-chunk-18

DENSE RESULTS
1 47 llama2-page-47-chunk-216
2 13 llama2-page-13-chunk-53
3 3 llama2-page-3-chunk-9
4 6 llama2-page-6-chunk-18


In [74]:
bm25_retriever.k = 8

dense_retriever = vector_store.as_retriever(
    search_type="similarity",
    search_kwargs={
        "k": 8
    },
)

## Hybrid Retriever
- It has both BM25 & Dense Retriever
- Weights are provided for each Retriver technique


In [75]:
from langchain_classic.retrievers.ensemble import EnsembleRetriever
hybrid_retriever = EnsembleRetriever(
    retrievers=[bm25_retriever, dense_retriever],
    weights=[0.5 #BM25 Weight
             , 0.5 #Dense Weight
             ]
)

In [76]:
hybrid_query = (
    "Llama 2 70B grouped-query attention and inference scalability"
)

In [78]:
hybrid_documents = hybrid_retriever.invoke(hybrid_query)

In [80]:
display_documents(
    hybrid_documents,
    title="Hybrid Retrieval: BM25 + Dense ",
    max_documents=6,
)


Hybrid Retrieval: BM25 + Dense 

RANK: 1
Paper page: 47
Section: appendix
Chunk ID: llama2-page-47-chunk-217
----------------------------------------------------------------------------------------------------
attention (MHA) models grow significantly. For larger models, where KV cache size becomes a bottleneck,
key and value projections can be shared across multiple heads without much degradation of performance
(Chowdheryetal.,2022). Eithertheoriginalmulti-queryformatwithasingleKVprojection(MQA, Shazeer,
2019) or a grouped-query attention variant with 8 KV projections (GQA, Ainslie et al., 2023) can be used.
In Table 18, we compare MQA and GQA variants with an MHA baseline. We train all models with 150B
tokens while keeping a fixed 30B model size. To keep a similar overall parameter count across GQ

RANK: 2
Paper page: 4
Section: introduction
Chunk ID: llama2-page-4-chunk-12
----------------------------------------------------------------------------------------------------
1. Llama 

In [86]:
test_query = (
    "How did Meta make Llama 2 70B efficient for large-scale inference?"
) 

sparse_results = bm25_retriever.invoke(test_query)
dense_results = dense_retriever.invoke(test_query)
hybrid_results = hybrid_retriever.invoke(test_query)

In [87]:
display_documents(
    sparse_results,
    title="1. Sparse Retrieval",
    max_documents=4,
)

display_documents(
    dense_results,
    title="2. Dense Retrieval",
    max_documents=4,
)

display_documents(
    hybrid_results,
    title="3. Hybrid Retrieval",
    max_documents=4,
)


1. Sparse Retrieval

RANK: 1
Paper page: 54
Section: appendix
Chunk ID: llama2-page-54-chunk-240
----------------------------------------------------------------------------------------------------
attribute, and so, up to 20 turns (we did not extend the human evaluation more, and all the examples had
less than 4048 tokens in total over the turns). As a comparison,Llama 2-Chat without GAtt can not anymore
refer to the attributes after only few turns: from 100% at turn t+1, to 10% at turn t+3 and then 0%.
GAtt Zero-shot Generalisation. We tried at inference time to set constrain not present in the training of
GAtt. For instance, “answer in one sentence only”, for which the model remained consistent, as illustrated in
Figure 28.
We applied first GAtt toLlama 1, which was pretrained with a 

RANK: 2
Paper page: 6
Section: pretraining
Chunk ID: llama2-page-6-chunk-18
----------------------------------------------------------------------------------------------------
Training Data Params C

## Query Rewriting